# Agenda

1. Useful methods + attributes
2. `NaN` and data frames
3. Boolean indexes and data frames
4. Adding columns
5. (Finally!) Files


In [1]:
import pandas as pd
from pandas import Series, DataFrame

In [2]:
df = DataFrame([[10, 20, 30],
                [40, 50, 60],
                [70, 80, 90],
                [100, 110, 120]],
               index=list('abcd'),
               columns=list('xyz'))
df

,x,y,z
a,10,20,30
b,40,50,60
c,70,80,90
d,100,110,120


In [3]:
# if I want the mean of a given column, I can grab that column and run mean

df['x'].mean()

np.float64(55.0)

In [4]:
# what happens if I invoke the mean method on the data frame, rather than one column?

df.mean()

x    55.0
y    65.0
z    75.0
dtype: float64

In [5]:
df['z'].max()

np.int64(120)

In [6]:
df.max()

x    100
y    110
z    120
dtype: int64

The general rule of thumb is that if you run a series method on a data frame, the method is run on each column, and you get a result for each column, labeled with the column's name.

However... if you want, you can often invoke the method passing the "axis" keyword argument. If you pass "axis='columns'", then it'll go the other way.

In [7]:
df

,x,y,z
a,10,20,30
b,40,50,60
c,70,80,90
d,100,110,120


In [8]:
df.max()

x    100
y    110
z    120
dtype: int64

In [10]:
df.max(axis='columns')

a     30
b     60
c     90
d    120
dtype: int64

In [11]:
df.max(axis=0)  # in NumPy, you have to say 0 or 1 for the axis, and 0 is the "default" axis

x    100
y    110
z    120
dtype: int64

In [12]:
df.max(axis=1)

a     30
b     60
c     90
d    120
dtype: int64

# More general series->data frame method rule of thumb:

- If the series method returns a single value, the data frame method will return a series
- If the series method returns a series, the data frame method will return a data frame

In [13]:
df['x'].describe()  # this returns a series, info about our column

count      4.000000
mean      55.000000
std       38.729833
min       10.000000
25%       32.500000
50%       55.000000
75%       77.500000
max      100.000000
Name: x, dtype: float64

In [14]:
df.describe()  # now I'll get a data frame back, with one column per column in df -- it'll only show numeric columns by default

,x,y,z
count,4.000000,4.000000,4.000000
mean,55.000000,65.000000,75.000000
std,38.729833,38.729833,38.729833
min,10.000000,20.000000,30.000000
25%,32.500000,42.500000,52.500000
50%,55.000000,65.000000,75.000000
75%,77.500000,87.500000,97.500000
max,100.000000,110.000000,120.000000


# NaN values 

In a series, we have three methods we can use:

- `dropna` -- returns only non-NaN values
- `fillna(n)` -- returns a new series in which NaN is replaced with n
- `interpolate` -- returns a series where `NaN` is replaced by the mean of its neighbors or (more generally) a linear function

In a data frame:

- `dropna` -- returns a data frame, where any ROW with `NaN` is removed
- `fillna` -- returns a data frame, where all `NaN` values are replaced ... or you can pass it a series, and fill a different value for each column
- `interpolate` -- doesn't really change, but works per column

In [15]:
df

,x,y,z
a,10,20,30
b,40,50,60
c,70,80,90
d,100,110,120


In [17]:
df.loc['b', 'y'] = float('nan')
df.loc['a', 'z'] = float('nan')
df.loc['a', 'x'] = float('nan')
df.loc['d', 'y'] = float('nan')


In [18]:
df

,x,y,z
a,NaN,20.0,NaN
b,40.0,NaN,60.0
c,70.0,80.0,90.0
d,100.0,NaN,120.0


In [19]:
df.dropna()  

,x,y,z
c,70.0,80.0,90.0


In [20]:
# what if you want to keep some of the NaN values? Some columns are more important than others...

df.dropna(thresh=2)  # this means: If there are 2 good values, keep the column

,x,y,z
b,40.0,NaN,60.0
c,70.0,80.0,90.0
d,100.0,NaN,120.0


In [21]:
df.dropna(subset='x')  # only count 'x' when looking for NaN

,x,y,z
b,40.0,NaN,60.0
c,70.0,80.0,90.0
d,100.0,NaN,120.0


In [22]:
df

,x,y,z
a,NaN,20.0,NaN
b,40.0,NaN,60.0
c,70.0,80.0,90.0
d,100.0,NaN,120.0


In [23]:
df.fillna(999)   # simple version of filling in values instead of NaN -- give a scalar value

,x,y,z
a,999.0,20.0,999.0
b,40.0,999.0,60.0
c,70.0,80.0,90.0
d,100.0,999.0,120.0


In [24]:
df.dtypes

x    float64
y    float64
z    float64
dtype: object

In [25]:
# it's very common to use fillna with the mean of each column
# we can do that by invoking mean, which will give a series
# then we can pass that series to fillna, and each column will be fillna'd with its own mean

df.mean()

x    70.0
y    50.0
z    90.0
dtype: float64

In [27]:
df

,x,y,z
a,NaN,20.0,NaN
b,40.0,NaN,60.0
c,70.0,80.0,90.0
d,100.0,NaN,120.0


In [28]:
df.fillna(df.mean())

,x,y,z
a,70.0,20.0,90.0
b,40.0,50.0,60.0
c,70.0,80.0,90.0
d,100.0,50.0,120.0


In [29]:
df.interpolate()

,x,y,z
a,NaN,20.0,NaN
b,40.0,50.0,60.0
c,70.0,80.0,90.0
d,100.0,80.0,120.0


In [30]:
help(df.interpolate)

Help on method interpolate in module pandas.core.generic:

interpolate(
    method: InterpolateOptions = 'linear',
    *,
    axis: Axis = 0,
    limit: int | None = None,
    inplace: bool = False,
    limit_direction: Literal['forward', 'backward', 'both'] | None = None,
    limit_area: Literal['inside', 'outside'] | None = None,
    **kwargs
) -> Self method of pandas.DataFrame instance
    Fill NaN values using an interpolation method.

    Please note that only ``method='linear'`` is supported for
    DataFrame/Series with a MultiIndex.

    Parameters
    ----------
    method : str, default 'linear'
        Interpolation technique to use. One of:

        * 'linear': Ignore the index and treat the values as equally
          spaced. This is the only method supported on MultiIndexes.
        * 'time': Works on daily and higher resolution data to interpolate
          given length of interval. This interpolates values based on
          time interval between observations.
        

# Adding columns

Very often, you'll want to add one or more columns to a data frame. Sometimes, it's for helpful temp storing of data while you're doing a complex set of calculations. Sometimes, you actually want to keep it around.

In either case, you can add a new column to a data frame by just assigning to it.

In [31]:
df

,x,y,z
a,NaN,20.0,NaN
b,40.0,NaN,60.0
c,70.0,80.0,90.0
d,100.0,NaN,120.0


In [32]:
df['t'] = 'hello to everyone today'.split()  # it must have 4 elements, otherwise assignment won't work

In [33]:
df['t']

a       hello
b          to
c    everyone
d       today
Name: t, dtype: str

In [34]:
df

,x,y,z,t
a,NaN,20.0,NaN,hello
b,40.0,NaN,60.0,to
c,70.0,80.0,90.0,everyone
d,100.0,NaN,120.0,today


In [36]:
df['product'] = df['x'] * df['z']
df['product']

a        NaN
b     2400.0
c     6300.0
d    12000.0
Name: product, dtype: float64

In [37]:
df

,x,y,z,t,product
a,NaN,20.0,NaN,hello,NaN
b,40.0,NaN,60.0,to,2400.0
c,70.0,80.0,90.0,everyone,6300.0
d,100.0,NaN,120.0,today,12000.0


# Setting and resetting the index

We've seen that retrieving values via the index is everywhere in Pandas. Sometimes, you'll want to set one of the "regular" columns to be the index. Or you'll want to "demote" a column from the index to be regular again.

- To make a column the index, use the `set_index` method. You have to provide a string, the name of the column.
- To remove a column from the index, turning it back into a regular column, use the `reset_index` method.

In [38]:
df

,x,y,z,t,product
a,NaN,20.0,NaN,hello,NaN
b,40.0,NaN,60.0,to,2400.0
c,70.0,80.0,90.0,everyone,6300.0
d,100.0,NaN,120.0,today,12000.0


In [40]:
df.reset_index()  # this returns a new data frame, and does not change df!

,index,x,y,z,t,product
0,a,NaN,20.0,NaN,hello,NaN
1,b,40.0,NaN,60.0,to,2400.0
2,c,70.0,80.0,90.0,everyone,6300.0
3,d,100.0,NaN,120.0,today,12000.0


In [41]:
df

,x,y,z,t,product
a,NaN,20.0,NaN,hello,NaN
b,40.0,NaN,60.0,to,2400.0
c,70.0,80.0,90.0,everyone,6300.0
d,100.0,NaN,120.0,today,12000.0


In [42]:
df.set_index('t')

,x,y,z,product
t,,,,
hello,NaN,20.0,NaN,NaN
to,40.0,NaN,60.0,2400.0
everyone,70.0,80.0,90.0,6300.0
today,100.0,NaN,120.0,12000.0


# Exercise: Groceries

1. Define/redefine a data frame with groceries. There should be three columns -- product, price, and department.
2. Set the index to be the product names.
3. Retrieve three items via the index, and find their min and max prices.
4. Set the index now to be the department. Find the mean price for items in any one department.

https://practice.lernerpython.com/classroom/ce06963a8d/ex-57

In [43]:
df = DataFrame([['apple', 5, 'produce'],
                ['bananas', 3, 'produce'],
                ['computer', 500, 'electronics'],
                ['diary', 20, 'books'],
                ['elephant', 100, 'pets']],
               index=[123, 124, 345, 456, 789],
               columns='product price department'.split())

df

,product,price,department
123,apple,5,produce
124,bananas,3,produce
345,computer,500,electronics
456,diary,20,books
789,elephant,100,pets


In [44]:
df.set_index('product')

,price,department
product,,
apple,5,produce
bananas,3,produce
computer,500,electronics
diary,20,books
elephant,100,pets


In [52]:
# method chaining

(
    df
    .set_index('product')
    .loc[ ['apple', 'computer', 'elephant'] ]   # fancy indexing
    .agg(['min', 'max'])
)

,price,department
min,5,electronics
max,500,produce


In [56]:
(
    df
    .set_index('department')
    .loc['produce']
    ['price']
    .mean()
)

np.float64(4.0)

# Exercise: Adding/modifying columns

1. Use the grocery store data frame.
2. Add a new column, `tax`, the percentage tax applied to each item. Vary according to the department.
3. Add a new column, `total_price`, which is the price + (price * tax).
4. Retrieve products where total_price is > 50.

https://practice.lernerpython.com/classroom/ce06963a8d/ex-58

In [57]:
df

,product,price,department
123,apple,5,produce
124,bananas,3,produce
345,computer,500,electronics
456,diary,20,books
789,elephant,100,pets


In [58]:
df['tax'] = 1.00  # this will be assigned to all rows

df

,product,price,department,tax
123,apple,5,produce,1.0
124,bananas,3,produce,1.0
345,computer,500,electronics,1.0
456,diary,20,books,1.0
789,elephant,100,pets,1.0


In [59]:
df['tax'] = [0.05, .05, .1, .01, 1.0]

df

,product,price,department,tax
123,apple,5,produce,0.05
124,bananas,3,produce,0.05
345,computer,500,electronics,0.10
456,diary,20,books,0.01
789,elephant,100,pets,1.00


In [61]:
# calculate and add to our data frame
df['total_price'] = df['price'] * df['tax']

In [62]:
df

,product,price,department,tax,total_price
123,apple,5,produce,0.05,0.25
124,bananas,3,produce,0.05,0.15
345,computer,500,electronics,0.10,50.00
456,diary,20,books,0.01,0.20
789,elephant,100,pets,1.00,100.00


In [64]:
(
    df
    .loc[ df['total_price'] >= 50 ]
)

,product,price,department,tax,total_price
345,computer,500,electronics,0.1,50.0
789,elephant,100,pets,1.0,100.0


# Loading real-world data! 

When I started to work with data, I was sure I would be using some binary file format that was really smart about assigning dtypes and moving them between the file and Pandas.

I was wrong -- we normally use CSV (comma-separated values, a text format) or Excel (mostly textual, but with some binary aspects) when we read data. There are some binary formats, especially Parquet (which is newish, and part of the PyArrow world), but mostly we work with CSV.

Normally, or in the simple case, a CSV file is:
- One record per line
- Fields in a record are separated with commas
- The first row in the file contains the headers, which will be used as the column names in our data frame
- Pandas examines the data in each column, and assigns a dtype:
    - If there are only digits, it assigns `int64`
    - If there are digits and a decimal point, it assigns `float64`
    - In all other cases, it treats the column as strings -- in Pandas 2, it'll be `object` and in Pandas 3, it'll be `str`.

To read a CSV file into Pandas:
1. Use the `pd.read_csv` method -- not `df.read_csv`, because we're creating a new data frame based on the file's contents.
2. We can pass the filename, and a number of other possible arguments.
3. The result, returned to us, is a new data frame.

In [66]:
filename = 'taxi.csv'
df = pd.read_csv(filename)  

df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,2,2015-06-02 11:19:29,2015-06-02 11:47:52,1,1.63,-73.954430,40.764141,1,N,-73.974754,40.754093,2,17.0,0.0,0.5,0.00,0.0,0.3,17.80
1,2,2015-06-02 11:19:30,2015-06-02 11:27:56,1,0.46,-73.971443,40.758942,1,N,-73.978539,40.761909,1,6.5,0.0,0.5,1.00,0.0,0.3,8.30
2,2,2015-06-02 11:19:31,2015-06-02 11:30:30,1,0.87,-73.978111,40.738434,1,N,-73.990273,40.745438,1,8.0,0.0,0.5,2.20,0.0,0.3,11.00
3,2,2015-06-02 11:19:31,2015-06-02 11:39:02,1,2.13,-73.945892,40.773529,1,N,-73.971527,40.760330,1,13.5,0.0,0.5,2.86,0.0,0.3,17.16
4,1,2015-06-02 11:19:32,2015-06-02 11:32:49,1,1.40,-73.979088,40.776772,1,N,-73.982162,40.758999,2,9.5,0.0,0.5,0.00,0.0,0.3,10.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,1,2015-06-01 00:12:59,2015-06-01 00:24:18,1,2.70,-73.947792,40.814972,1,N,-73.973358,40.783638,2,11.0,0.5,0.5,0.00,0.0,0.3,12.30
9995,1,2015-06-01 00:12:59,2015-06-01 00:28:16,1,4.50,-74.004066,40.747818,1,N,-73.953758,40.779285,1,16.0,0.5,0.5,3.00,0.0,0.3,20.30
9996,2,2015-06-01 00:13:00,2015-06-01 00:37:25,1,5.59,-73.994377,40.766102,1,N,-73.903206,40.750546,2,21.0,0.5,0.5,0.00,0.0,0.3,22.30
9997,2,2015-06-01 00:13:02,2015-06-01 00:19:10,6,1.54,-73.978302,40.748531,1,N,-73.989166,40.762852,2,6.5,0.5,0.5,0.00,0.0,0.3,7.80


In [67]:
# how many parameters does read_csv support?

help(pd.read_csv)

Help on function read_csv in module pandas:

read_csv(
    filepath_or_buffer: FilePath | ReadCsvBuffer[bytes] | ReadCsvBuffer[str],
    *,
    sep: str | None | lib.NoDefault = <no_default>,
    delimiter: str | None | lib.NoDefault = None,
    header: int | Sequence[int] | None | Literal['infer'] = 'infer',
    names: Sequence[Hashable] | None | lib.NoDefault = <no_default>,
    index_col: IndexLabel | Literal[False] | None = None,
    usecols: UsecolsArgType = None,
    dtype: DtypeArg | None = None,
    engine: CSVEngine | None = None,
    converters: Mapping[HashableT, Callable] | None = None,
    true_values: list | None = None,
    false_values: list | None = None,
    skipinitialspace: bool = False,
    skiprows: list[int] | int | Callable[[Hashable], bool] | None = None,
    skipfooter: int = 0,
    nrows: int | None = None,
    na_values: Hashable | Iterable[Hashable] | Mapping[Hashable, Iterable[Hashable]] | None = None,
    keep_default_na: bool = True,
    na_filter: bool 

# I've got the data. How do we work with it?

Our favorite tool to retrieve selected parts of the data is to use `.loc`.

We can use `.loc` on a data frame with either one argument (selecting the rows) or with two arguments (selecting the rows, and then selecting the columns):

- `.loc` with one argument, it can be
    - a single index
    - a list of indexes
    - a slice
    - a boolean series
- when `.loc` has two arguments, the second is the column selector:
    - a single column name
    - a list of columns names
    - (very rarely) a boolean series for selective retrieval


In [76]:
# I want the mean total_price for all rides where trip_distance was > 20.

(
    df
    .loc[
            pd.col('trip_distance') > 20    # row selector
              ,
            'total_amount'                   # column selector
        ]
    .mean()
)

np.float64(82.8916)

# Exercise: Weird taxi rides

1. Read `taxi.csv` into a data frame. This contains 9,999 taxi rides + one header row.
2. How many rides went 0 miles (`trip_distance`)? How much, on average, did people pay (`total_amount`) for such rides?
3. How many rides cost <= 0 dollars? (`total_amount`) How far (`trip_distance`), on average, did people go on such trips?
4. How many rides had 0 `passenger_count`? What were the mean `trip_distance` and `total_amount` for those rides?

https://practice.lernerpython.com/classroom/ce06963a8d/ex-59

In [77]:
df = pd.read_csv('taxi.csv')

In [82]:
# How many rides went 0 miles (trip_distance)? How much, on average, did people pay (total_amount) for such rides?

(
    df
    .loc[ pd.col('trip_distance') == 0 ,  # row selector
          'total_amount' ]   # column selector
    .agg(['mean', 'count'])
)

mean     31.58194
count    67.00000
Name: total_amount, dtype: float64

In [ ]:
# How many rides cost <= 0 dollars? (total_amount) How far (trip_distance), on average, did people go on such trips?
# How many rides had 0 passenger_count? What were the mean trip_distance and total_amount for those rides?